In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys
sys.path.append('../')

### Change colours to use the new ones

In [ ]:
from constants import brussels_muni_codes_dict, brussels_codes_muni_dict, PT_COLOR, BIKE_COLOR

In [ ]:
PURPLE = PT_COLOR
ORANGE = BIKE_COLOR

In [ ]:
municipalities = [str(code) for code in brussels_muni_codes_dict.values()]
municipality_codes = [21001, 21002, 21003, 21004, 21005, 21006, 21007, 21008, 21009, 21010, 21011, 21012, 21013, 21014, 21015, 21016, 21017, 21018, 21019]

# MUNICIPALITY LEVEL

In [ ]:
# Load data
workers = pd.read_csv("output/workers_with_work_addresses.csv")

db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT 
    CD_SECTOR_RESIDENCE AS home_statistical_sector,
    CD_SECTOR_WORK AS work_statistical_sector, 
    CAST(OBS_VALUE AS INTEGER) AS num_people
FROM 'VU_GEO_LPW_MATRIX'
WHERE CD_RGN_RESIDENCE = '04000'
AND CD_SECTOR_RESIDENCE NOT LIKE '%ZZZZ'
AND CD_SECTOR_WORK NOT LIKE '%ZZZZ'
AND CD_SECTOR_WORK NOT LIKE 'FOR'
"""
od_matrix = pd.read_sql_query(query, conn)
conn.close()

# Extract the part after underscore for sector codes
od_matrix["home_statistical_sector"] = od_matrix["home_statistical_sector"].str.split("_").str[1]
od_matrix["work_statistical_sector"] = od_matrix["work_statistical_sector"].str.split("_").str[1]

In [ ]:
# Extract municipality codes (first 5 characters)
od_matrix["work_municipality"] = od_matrix["work_statistical_sector"].str[:5]
workers["work_municipality"] = workers["work_sector"].str[:5]

# Group od_matrix by work municipality
od_by_municipality = od_matrix.groupby("work_municipality")["num_people"].sum().reset_index()
od_by_municipality.columns = ["municipality", "od_matrix_count"]

# Group workers by work municipality
workers_by_municipality = workers.groupby("work_municipality").size().reset_index(name="workers_count")

# Merge and compare
comparison = od_by_municipality.merge(workers_by_municipality, left_on="municipality", right_on="work_municipality", how="outer")

print(comparison[comparison["municipality"].astype(int).isin(municipality_codes)].head())

# 2021 MUNICIPALITY LEVEL

In [ ]:
df_raw = pd.read_csv('input_data/working_population_home_work_municipality_2021_BCR_all.csv')

In [ ]:
# Reshape df_raw to long format for analysis
df_2021 = df_raw.copy()

# Set the index to 'Code INS' and extract municipality codes
df_2021 = df_2021.set_index('Code INS')
df_2021 = df_2021.drop('Lieu de résidence', axis=1)

# Melt to long format
df_2021_long = df_2021.reset_index().melt(
    id_vars='Code INS',
    var_name='work_municipality_code',
    value_name='count'
)

# Rename columns for consistency
df_2021_long = df_2021_long.rename(columns={
    'Code INS': 'home_muni_code',
    'work_municipality_code': 'work_muni_code',
    'count': 'num_people'
})

# Convert to numeric
df_2021_long['home_muni_code'] = pd.to_numeric(df_2021_long['home_muni_code'], errors='coerce')
df_2021_long['work_muni_code'] = pd.to_numeric(df_2021_long['work_muni_code'], errors='coerce')

# Filter for valid Brussels municipalities
df_2021_long = df_2021_long[
    df_2021_long['home_muni_code'].isin(municipality_codes) &
    df_2021_long['work_muni_code'].isin(municipality_codes)
]

print(df_2021_long.head())

In [ ]:
# Increase factor to account for the people travelling to 04000_ZZZZZZ
increase_factor = 1.0257

# Build 2021 municipality totals (Brussels -> Brussels), aligned to BE100 codes
od_2021_by_work = (
    df_2021_long.groupby("work_muni_code", as_index=False)["num_people"]
    .sum()
    .assign(municipality=lambda d: d["work_muni_code"].astype(int).astype(str))
)

# Keep same municipality order as existing BE100 plot
plot_2021 = (
    pd.DataFrame({"municipality": municipalities})
    .merge(od_2021_by_work[["municipality", "num_people"]], on="municipality", how="left")
    .merge(workers_by_municipality, left_on="municipality", right_on="work_municipality", how="left")
    .merge(od_by_municipality, on="municipality", how="left")
    .fillna(0)
)

def format_compact(x, pos=None):
    x = float(x)
    ax = abs(x)

    if ax >= 1_000_000:
        v = x / 1_000_000
        return f"{v:.1f}".rstrip("0").rstrip(".") + "M"
    if ax >= 1_000:
        v = x / 1_000
        return f"{v:.1f}".rstrip("0").rstrip(".") + "k"
    return f"{int(x):,}"


with plt.rc_context({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "figure.titlesize": 22,
}):
    # Plot: Synthetic vs Census 2021 vs Census 2011
    fig, ax = plt.subplots(figsize=(14, 8))
    pos = np.arange(len(plot_2021))
    bar_w = 0.25

    ax.bar(pos - bar_w, plot_2021["workers_count"], bar_w, label="Synthetic Population", color=PURPLE, alpha=1.0)
    ax.bar(pos, plot_2021["num_people"]*increase_factor, bar_w, label="Census (2021)", color=PURPLE, alpha=0.5)
    ax.bar(pos + bar_w, plot_2021["od_matrix_count"], bar_w, label="Census (2011)", color=PURPLE, alpha=0.3)

    # Right axis: relative error between synthetic and 2021 census (%)
    ax2 = ax.twinx()

    syn = plot_2021["workers_count"].to_numpy(dtype=float)
    cen = plot_2021["num_people"].to_numpy(dtype=float) * increase_factor
    rel_err = np.where(cen > 0, 100.0 * (syn - cen) / cen, np.nan)

    ax2.plot(pos, rel_err, linestyle="--", marker="o", linewidth=1.5,
            markersize=4, color=ORANGE, label="Rel. error synth vs 2021 (%)")
    ax2.axhline(0, color="gray", linewidth=1, alpha=0.7)
    ax2.set_ylabel("Relative error (%)")

    codes_2021 = plot_2021["municipality"].str[:5].astype(int)
    labels_2021 = [brussels_codes_muni_dict.get(c, f"Unknown ({c})") for c in codes_2021]

    ax.set_xticks(pos)
    ax.set_xticklabels(labels_2021, rotation=45, ha="right")
    ax.set_xlabel("Municipality")
    ax.set_ylabel("Population")
    ax.set_title("Number of Brussels residents by work municipality: Synthetic vs Census 2021 vs Census 2011")
    ax.grid(axis="y", alpha=0.3)

    # Combine legends from both axes
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_compact))

    plt.tight_layout()
    plt.savefig("figures/validation_work_municipality_comparison_rel_error.png", dpi=300)
    plt.show()

# MUNICIPALITY TO MUNICIPALITY HEATMAP

In [ ]:
# Municipality-to-municipality OD heatmaps (Synthetic vs Census)

# Build synthetic OD matrix: home municipality -> work municipality
synthetic = workers.copy()
synthetic["home_municipality"] = synthetic["sector"].str[:5]
synthetic["work_municipality"] = synthetic["work_municipality"].astype(str)

synthetic = synthetic[
    synthetic["home_municipality"].isin(municipalities) &
    synthetic["work_municipality"].isin(municipalities)
]

synthetic_od = (
    synthetic.groupby(["home_municipality", "work_municipality"])
    .size()
    .reset_index(name="num_people")
)

synthetic_pivot = (
    synthetic_od.pivot(index="home_municipality", columns="work_municipality", values="num_people")
    .reindex(index=municipalities, columns=municipalities)
    .fillna(0)
)

# Build 2021 census OD matrix
census_2021_pivot = (
    df_2021_long.pivot(index="home_muni_code", columns="work_muni_code", values="num_people")
    .reindex(index=municipality_codes, columns=municipality_codes)
    .fillna(0)
)

# Common scale for both heatmaps
vmax = max(synthetic_pivot.to_numpy().max(), census_2021_pivot.to_numpy().max())

labels = [brussels_codes_muni_dict.get(code, str(code)) for code in municipality_codes]

with plt.rc_context({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "figure.titlesize": 22,
}):
    fig, axes = plt.subplots(1, 2, figsize=(18, 9), sharex=True, sharey=True)

    im0 = axes[0].imshow(synthetic_pivot.to_numpy(), cmap="inferno_r", aspect="auto", vmin=0, vmax=vmax)
    axes[0].set_title("Synthetic population")
    axes[0].set_xticks(np.arange(len(labels)))
    axes[0].set_yticks(np.arange(len(labels)))
    axes[0].set_xticklabels(labels, rotation=45, ha="right")
    axes[0].set_yticklabels(labels)
    axes[0].set_xlabel("Work municipality")
    axes[0].set_ylabel("Home municipality")

    im1 = axes[1].imshow(census_2021_pivot.to_numpy(), cmap="inferno_r", aspect="auto", vmin=0, vmax=vmax)
    axes[1].set_title("Census 2021")
    axes[1].set_xticks(np.arange(len(labels)))
    axes[1].set_xticklabels(labels, rotation=45, ha="right")
    axes[1].set_yticks(np.arange(len(labels)))
    axes[1].set_yticklabels(labels)
    axes[1].set_xlabel("Work municipality")

    fig.subplots_adjust(right=0.90)
    cax = fig.add_axes([0.91, 0.27, 0.02, 0.6])  # [left, bottom, width, height]
    fig.colorbar(im1, cax=cax, label="Number of people")

    fig.suptitle("Brussels municipality-to-municipality commuting flows", y=0.99)
    plt.tight_layout(rect=(0, 0, 0.90, 1))  # keep space for colorbar
    plt.savefig("figures/validation_work_municipality_od_heatmaps.png", dpi=300)
    plt.show()

In [ ]:
# Municipality-to-municipality OD heatmaps (Synthetic vs Census)

# Build synthetic OD matrix: home municipality -> work municipality
synthetic = workers.copy()
synthetic["home_municipality"] = synthetic["sector"].str[:5]
synthetic["work_municipality"] = synthetic["work_municipality"].astype(str)

synthetic = synthetic[
    synthetic["home_municipality"].isin(municipalities) &
    synthetic["work_municipality"].isin(municipalities)
]

synthetic_od = (
    synthetic.groupby(["home_municipality", "work_municipality"])
    .size()
    .reset_index(name="num_people")
)

synthetic_pivot = (
    synthetic_od.pivot(index="home_municipality", columns="work_municipality", values="num_people")
    .reindex(index=municipalities, columns=municipalities)
    .fillna(0)
)
synthetic_pivot = synthetic_pivot.div(synthetic_pivot.sum(axis=1), axis=0).fillna(0) * 100

# Build 2021 census OD matrix
census_2021_pivot = (
    df_2021_long.pivot(index="home_muni_code", columns="work_muni_code", values="num_people")
    .reindex(index=municipality_codes, columns=municipality_codes)
    .fillna(0)
)
census_2021_pivot = census_2021_pivot.div(census_2021_pivot.sum(axis=1), axis=0).fillna(0) * 100

# Common scale for both heatmaps
vmax = 80

labels = [brussels_codes_muni_dict.get(code, str(code)) for code in municipality_codes]

with plt.rc_context({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "figure.titlesize": 22,
}):
    fig, axes = plt.subplots(1, 2, figsize=(18, 9), sharex=True, sharey=True)

    im0 = axes[0].imshow(synthetic_pivot.to_numpy(), cmap="inferno_r", aspect="auto", vmin=0, vmax=vmax)
    axes[0].set_title("Synthetic population")
    axes[0].set_xticks(np.arange(len(labels)))
    axes[0].set_yticks(np.arange(len(labels)))
    axes[0].set_xticklabels(labels, rotation=45, ha="right")
    axes[0].set_yticklabels(labels)
    axes[0].set_xlabel("Work municipality")
    axes[0].set_ylabel("Home municipality")

    im1 = axes[1].imshow(census_2021_pivot.to_numpy(), cmap="inferno_r", aspect="auto", vmin=0, vmax=vmax)
    axes[1].set_title("Census 2021")
    axes[1].set_xticks(np.arange(len(labels)))
    axes[1].set_xticklabels(labels, rotation=45, ha="right")
    axes[1].set_yticks(np.arange(len(labels)))
    axes[1].set_yticklabels(labels)
    axes[1].set_xlabel("Work municipality")

    fig.subplots_adjust(right=0.90)
    cax = fig.add_axes([0.91, 0.27, 0.02, 0.6])  # [left, bottom, width, height]
    fig.colorbar(im1, cax=cax, label="Share of people within home municipality (%)")

    fig.suptitle("Brussels municipality-to-municipality commuting flows", y=0.99)
    plt.tight_layout(rect=(0, 0, 0.90, 1))  # keep space for colorbar
    plt.savefig("figures/validation_work_municipality_od_heatmaps_pct.png", dpi=300)
    plt.show()